# Run Nori-Rel on one RelBench task

This notebook walks through the complete Nori-Rel workflow: verify the environment, choose a supported regression task, precompute its relational features, run the frozen public Nori 30M checkpoint, and save the result.

Nori-Rel performs in-context regression over depth-2 Deep Feature Synthesis (DFS) features. It does **not** fine-tune the checkpoint. The adapter is regression-only and deliberately disables silent context subsampling and cache quantization.

> **Resources:** an NVIDIA GPU is strongly recommended. DFS runs on CPU and can be slow; larger datasets can also need substantial host RAM while the BF16 context cache is active. The first run downloads the selected RelBench database and the public checkpoint.

## Step 1 — install and launch the notebook

Use Python 3.11 or 3.12. From a RelArena source checkout, launch Jupyter in the project environment:

```bash
uv sync --extra nori-rel
uv run --extra nori-rel --with jupyter jupyter lab examples/nori_rel.ipynb
```

If Jupyter is already running from that environment, continue with the next cell.

In [ ]:
import sys
from importlib.util import find_spec

import torch

assert sys.version_info[:2] in {(3, 11), (3, 12)}, sys.version
assert find_spec("synthefy_nori") is not None, (
    "The nori-rel extra is missing. Relaunch with: "
    "uv run --extra nori-rel --with jupyter jupyter lab"
)

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | {device}")
if not torch.cuda.is_available():
    print("Warning: Nori-Rel can run on CPU, but a GPU is strongly recommended.")

## Step 2 — choose storage and a task

The example uses `rel-f1/driver-position`, a regression task. Change `DATASET` and `TASK` to another supported pair listed in the next step. Keep `SEED = 0` and `N_TRIALS = 1` to reproduce the submitted fixed configuration.

Feature artifacts, downloaded data, and the checkpoint are kept outside the repository. Set `WORK_DIR` to a fast disk with enough free space.

In [ ]:
import os
from pathlib import Path

DATASET = "rel-f1"
TASK = "driver-position"
SEED = 0
N_TRIALS = 1

WORK_DIR = Path.home() / ".cache" / "relarena" / "nori-rel"
FEATURE_CACHE = WORK_DIR / "features"
RESULTS_DIR = Path("results") / "nori-rel"

os.environ.setdefault("HF_HOME", str(WORK_DIR / "huggingface"))
os.environ.setdefault("RELBENCH_CACHE_DIR", str(WORK_DIR / "relbench"))
FEATURE_CACHE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Feature cache: {FEATURE_CACHE}")
print(f"Results:       {RESULTS_DIR.resolve()}")

## Step 3 — confirm the task is supported

Nori-Rel supports regression only. Listing the eligible RelBench tasks reads registry metadata and does not download any datasets.

In [ ]:
import pandas as pd
from relbench.base import TaskType

from relarena import list_entity_tasks

regression_tasks = list_entity_tasks(task_types=frozenset({TaskType.REGRESSION}))
task_table = pd.DataFrame(
    [(spec.dataset, spec.task) for spec in regression_tasks],
    columns=["dataset", "task"],
)
assert (DATASET, TASK) in {(spec.dataset, spec.task) for spec in regression_tasks}, (
    f"{DATASET}/{TASK} is not a supported regression task"
)
task_table

## Step 4 — warm the DFS feature cache

RelArena must build leak-safe depth-2 relational features for the inner validation split and both outer-fit histories. This is the CPU-heavy stage. Run it once per dataset/task/cache combination; later model runs read the stored Parquet artifacts.

If someone has already provided a compatible warmed cache, point `FEATURE_CACHE` at it and skip this cell. A cache miss during the experiment is an error by design.

In [ ]:
from relarena import CacheConfig, RelBenchDatasetTask
from relarena.featurization.warm_cache import warm_dfs_cache

source = RelBenchDatasetTask(DATASET, TASK)
assert source.task.task_type is TaskType.REGRESSION
warm_dfs_cache(
    source,
    CacheConfig(FEATURE_CACHE, on_miss="fill"),
    max_depth=2,
)
print(f"DFS cache ready at {FEATURE_CACHE}")

## Step 5 — run Nori-Rel

Importing `relarena.models` registers the built-in models. `run_experiment` then runs the nested temporal protocol: fit on train and score validation, select the fixed configuration, refit on train + validation, and evaluate once on the hidden test split.

The checkpoint downloads and its SHA-256 is verified on the first fit. Run only one Nori-Rel task per GPU process.

In [ ]:
import relarena.models  # noqa: F401 — registers Nori-Rel and the other models
from relarena import registry, run_experiment

model_cls = registry.get("nori-rel")
summary = run_experiment(
    model_cls,
    DATASET,
    TASK,
    seed=SEED,
    n_trials=N_TRIALS,
    cache_dir=FEATURE_CACHE,
    cache_predictions=True,
)
summary

## Step 6 — inspect the result

The result table contains every evaluated configuration, validation and test metrics, and separate fit/predict timings. Nori-Rel has one fixed configuration, so the single row is both the default and selected result. For MAE, lower is better.

In [ ]:
from relarena import summary_to_dataframe

results = summary_to_dataframe(summary)
headline_columns = [
    column
    for column in (
        "model",
        "dataset",
        "task",
        "metric",
        "selected",
        "val_score",
        "test_score",
        "fit_time_tuning",
        "predict_time_tuning",
        "fit_time_refit",
        "predict_time_refit",
    )
    if column in results.columns
]
results[headline_columns]

## Step 7 — save the reproducible CSV

The CSV follows RelArena's shared results schema and can be concatenated with other task runs before leaderboard analysis.

In [ ]:
output_path = RESULTS_DIR / f"{DATASET}-{TASK}-seed-{SEED}.csv"
results.to_csv(output_path, index=False)
print(f"Saved {output_path.resolve()}")

## Command-line equivalent

After choosing the same paths, these commands perform the cache warm and experiment without Jupyter:

```bash
uv run --extra nori-rel python -m relarena.featurization.warm_cache \
  --dataset rel-f1 --task driver-position \
  --cache-dir ~/.cache/relarena/nori-rel/features --max-depth 2

CUDA_VISIBLE_DEVICES=0 uv run --extra nori-rel relarena \
  --model nori-rel --datasets rel-f1 --tasks driver-position \
  --seed 0 --n-trials 1 \
  --cache-dir ~/.cache/relarena/nori-rel/features \
  --output results/nori-rel/rel-f1-driver-position-seed-0.csv
```

### Troubleshooting

- **Unsupported task type:** choose a row from the regression task table in Step 3; classification is intentionally unsupported.
- **Cache miss:** rerun Step 4 with the same `DATASET`, `TASK`, and `FEATURE_CACHE`.
- **CUDA out of memory:** do not run concurrent tasks on one GPU. The adapter already bounds each forward pass and uses a host-offloaded BF16 cache without lossy fallbacks.
- **Slow first run:** DFS is CPU-heavy and the initial run downloads data and checkpoint files. Reusing `FEATURE_CACHE`, `HF_HOME`, and `RELBENCH_CACHE_DIR` avoids repeating that work.